In [1]:
#0: Gameplan sequence

#Pre-game actions: Iterative London mulligan (with card type requirements)

# Turn 0: 
# Draw 7 (Cards in hand = 7; Total cards seen = 7)
# Do mulligan once. If not fulfilling requirements, do it again. Regardless of the second result, go ahead.

# Turn 1
# Draw 1 (Cards in hand = 8; Total cards seen = 8)
# Play a land (Cards in hand = 7; Total cards seen = 8)
# If available, play a 1CMC draw spell. If so, add 1 to the number of seen cards onwards.

# Turn 2
# Draw 1 (Cards in hand = 8; Total cards seen = 9)
# Play a land (Cards in hand = 7; Total cards seen = 9)
# Play a ramp piece (Cards in hand = 6; Total cards seen = 9)

#Turn 3:
# Draw 1 (Cards in hand = 7; Total cards seen = 10)
# Play A&N (no effect in hand)

#Turn 4:
# Draw 1 (Cards in hand = 8; Total cards seen = 11)
# Play Bomb (Cards in hand = 7; Total cards seen = 11)

In [7]:
#1: Function and global constant definitions
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations
from lib.simulation import simulate_games, CARD_TYPE_MAP
from lib.analysis import print_simulation_report, LRBD_grid_search, analyze_sensitivity, analyze_single_variable, fs
from pathlib import Path
#%matplotlib widget

N = 99 #always 99 cards in deck

# --- Game plan (per turn list of card types to play) ---
gameplan = [
    np.array([1, 4], dtype=np.uint8),   # turn 1: play land + optional draw
    np.array([1, 2], dtype=np.uint8),   # turn 2: play land and ramp
    np.array([1], dtype=np.uint8),      # turn 3: play land
    np.array([1, 3], dtype=np.uint8)    # turn 4: play land and bomb
]

In [3]:
#2: Numba compilation of simulation function (dummy parameters)
L = 0      # lands
R = 0      # ramp
B = 0      # bombs
D = 0      # draw 
t_land = 0  # minimum lands to keep hand
t_ramp = 0  # minimum ramp to keep hand
t_bomb = 0  # minimum bombs to keep hand
t_draw = 0  # minimum draw to keep hand
N_mulligans = 5
user_priority = ("O", "B", "D", "R", "L")

# --- Build the deck template ---
base_deck = np.concatenate([
    np.ones(L, dtype=np.uint8),             # land card type = 1
    np.full(R, 2, dtype=np.uint8),          # ramp card type = 2
    np.full(B, 3, dtype=np.uint8),          # bomb card type = 3
    np.full(D, 4, dtype=np.uint8),          # draw card type = 4
    np.zeros(N - L - R - B - D, dtype=np.uint8) # filler / other cards = 0
])

# --- Run simulation once (for Numba compilation) ---
numeric_priority = tuple(CARD_TYPE_MAP[c] for c in user_priority)
result, fails, mulligans = simulate_games(1, base_deck, t_land, t_ramp, t_bomb, t_draw, gameplan, 
                                          N_mulligans, numeric_priority)
print("Compilation OK")

Compilation OK


In [ ]:
#3: Run single configuration game simulation with user input parameters
L = 36 # number of lands
R = 15 # number of ramp pieces 
B = 17 # number of bombs
D = 5 # number of cmc=1 draw spells
t_land = 2  # minimum lands to keep hand
t_ramp = 1  # minimum ramp to keep hand
t_bomb = 0  # minimum bombs to keep hand
t_draw = 0
N_sim = 1_000_000  # number of simulations
N_mulligans = 4
user_priority = ('O', 'R', 'D', 'B', 'L')

# --- Build the deck template ---
base_deck = np.concatenate([
    np.ones(L, dtype=np.uint8),             # land card type = 1
    np.full(R, 2, dtype=np.uint8),          # ramp card type = 2
    np.full(B, 3, dtype=np.uint8),          # bomb card type = 3
    np.full(D, 4, dtype=np.uint8),          # draw card type = 4
    np.zeros(N - L - R - B - D, dtype=np.uint8) # filler / other cards = 0
])

numeric_priority = tuple(CARD_TYPE_MAP[c] for c in user_priority)
result, fail_summary, mulligan_stats = simulate_games(N_sim, base_deck, t_land, t_ramp, t_bomb, t_draw, gameplan,
                                                 N_mulligans, numeric_priority)
print_simulation_report(result, fail_summary, mulligan_stats, N_sim, N_mulligans)

In [12]:
#4: Optimizing mulligan: Which is the best bottoming priority?
card_types = tuple(CARD_TYPE_MAP.keys()) #sample tuple with all card types
all_priorities = list(permutations(user_priority)) #calculate all permutations: 5! = 120 possible priorities
result = [] #empty list to store results

L = 36 # number of lands
R = 15 # number of ramp pieces 
B = 17 # number of bombs
D = 5 # number of cmc=1 draw spells
t_land = 2  # minimum lands to keep hand
t_ramp = 1  # minimum ramp to keep hand
t_bomb = 0  # minimum bombs to keep hand
t_draw = 0
N_sim = 10_000_000  # number of simulations
N_mulligans = 4

# --- Build the deck template ---
base_deck = np.concatenate([
    np.ones(L, dtype=np.uint8),             # land card type = 1
    np.full(R, 2, dtype=np.uint8),          # ramp card type = 2
    np.full(B, 3, dtype=np.uint8),          # bomb card type = 3
    np.full(D, 4, dtype=np.uint8),          # draw card type = 4
    np.zeros(N - L - R - B - D, dtype=np.uint8) # filler / other cards = 0
])

for priority_combination in all_priorities:
    numeric_priority = tuple(CARD_TYPE_MAP[c] for c in priority_combination)
    winrate, fail_summary, mulligan_stats = simulate_games(N_sim, base_deck, t_land, t_ramp, t_bomb, t_draw, gameplan,
                                                 N_mulligans, numeric_priority)
    result.append((priority_combination, winrate))

#Find maximum winrate and associated priority
result_sorted = sorted(result, key=lambda x: x[1], reverse=True) #sort by winrate

# --- Build plot data ---
labels = ["".join(p) for p, _ in result_sorted]
winrates = [w for _, w in result_sorted]
x = np.arange(len(labels))

# --- Plot ---
fig, ax = plt.subplots(figsize=(7.5, 2.5))
ax.bar(x, winrates, color="steelblue", width=0.7)
# Highlight the best priority
ax.bar(0, winrates[0], color="darkorange", width=0.7, label=f"Best: {labels[0]} ({winrates[0]:.2f}%)")
# Axis formatting
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, fontsize=3.5)
ax.set_xlabel("Bottoming priority (left = first to bottom)", fontsize=5)
ax.set_ylabel("Winrate (%)", fontsize=5)
ax.set_title("Winrate by mulligan bottoming priority (sorted best → worst)", fontsize=6)
ax.tick_params(axis='y', labelsize=4)
margin = (max(winrates) - min(winrates)) * 0.05
ax.set_ylim(min(winrates) - margin, max(winrates) + margin)
ax.legend(fontsize=4)
ax.grid(axis="y", alpha=0.4, linewidth=0.4)
ax.set_xlim(-0.5, len(labels) - 0.5)
plt.tight_layout(pad=0.3)
save_dir = Path.cwd() / "data"
plt.savefig(save_dir / "mulligan_priority.png", dpi=500, bbox_inches='tight')
plt.close()   

In [13]:
# Evaluate variation along each axis

fixed_config = {'L': 36, 'R': 15,'B': 17,'D': 5}
N_sim = 10_000_000
t_land = 2  # minimum lands to keep hand
t_ramp = 1  # minimum ramp to keep hand
t_bomb = 0  # minimum bombs to keep hand
t_draw = 0  # minimum 1cmc Draw spells to keep hand
N_mulligans = 4
priority = ("O", "R", "D", "B", "L")

values_L, winrates_L, deriv_L = analyze_single_variable('L', 33, 40,
                                           fixed_config, N,
                                           t_land, t_ramp, t_bomb, t_draw,
                                           N_sim, gameplan,
                                           N_mulligans, priority, save_plot=True)

values_B, winrates_B, deriv_B = analyze_single_variable('B', 10, 20,
                                           fixed_config, N,
                                           t_land, t_ramp, t_bomb, t_draw,
                                           N_sim, gameplan,
                                           N_mulligans, priority, save_plot=True)

values_R, winrates_R, deriv_R = analyze_single_variable('R', 10, 20,
                                           fixed_config, N,
                                           t_land, t_ramp, t_bomb, t_draw,
                                           N_sim, gameplan,
                                           N_mulligans, priority, save_plot=True)

values_D, winrates_D, deriv_D = analyze_single_variable('D', 0, 10,
                                           fixed_config, N,
                                           t_land, t_ramp, t_bomb, t_draw,
                                           N_sim, gameplan,
                                           N_mulligans, priority, save_plot=True)